In [5]:
import os
import shutil
from pathlib import Path

# Chemin absolu
base = Path('/home/medraedboukari/Desktop/AgriEdge-AI')

datasets = {
    'plant': base / 'data/roboflow_plant_disease',
    'tomato': base / 'data/roboflow_tomato',
    'cassava': base / 'data/roboflow_cassava2'
}

merged = base / 'data/merged'
counts = {'train': 0, 'valid': 0, 'test': 0}

for name, dataset_path in datasets.items():
    for split in ['train', 'valid', 'test']:
        img_src = dataset_path / split / 'images'
        lbl_src = dataset_path / split / 'labels'
        
        if not img_src.exists():
            print(f"⚠️ {name}/{split}/images introuvable")
            continue
            
        images = list(img_src.glob('*.jpg')) + list(img_src.glob('*.png'))
        
        for img in images:
            new_name = f"{name}_{img.name}"
            shutil.copy(img, merged / split / 'images' / new_name)
            lbl = lbl_src / img.with_suffix('.txt').name
            if lbl.exists():
                shutil.copy(lbl, merged / split / 'labels' / new_name.replace(img.suffix, '.txt'))
        
        counts[split] += len(images)
        print(f"✅ {name}/{split}: {len(images)} images copiées")

print(f"\n📊 Total fusionné:")
for split, count in counts.items():
    print(f"  {split}: {count} images")

✅ plant/train: 6052 images copiées
✅ plant/valid: 2014 images copiées
✅ plant/test: 1063 images copiées
✅ tomato/train: 12762 images copiées
✅ tomato/valid: 1374 images copiées
✅ tomato/test: 865 images copiées
✅ cassava/train: 9818 images copiées
✅ cassava/valid: 1506 images copiées
✅ cassava/test: 1505 images copiées

📊 Total fusionné:
  train: 28632 images
  valid: 4894 images
  test: 3433 images


In [6]:
yaml_content = """train: /home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/images
val: /home/medraedboukari/Desktop/AgriEdge-AI/data/merged/valid/images
test: /home/medraedboukari/Desktop/AgriEdge-AI/data/merged/test/images

nc: 24
names: [
  'Chlorosis Nutrient Deficiency', 'Water Excess', 'Sun Excess', 'Harmful Insects',
  'Water Deficiency', 'Sun Deficiency', 'Powdery Mildew', 'Parasites', 'Abnormal Redness',
  'Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold',
  'Leaf Miner', 'Mosaic Virus', 'Septoria', 'Spider Mites', 'Yellow Leaf Curl Virus',
  'CBB', 'CBSD', 'CGM', 'CMD', 'CASSAVA_HEALTHY'
]
"""

with open('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml créé !")

✅ data.yaml créé !


In [7]:
import os
from pathlib import Path

labels_path = Path('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/labels')
labels = list(labels_path.glob('*.txt'))

print(f"Nombre de fichiers labels : {len(labels)}")

# Vérifier un label exemple
with open(labels[0], 'r') as f:
    print(f"\nExemple label ({labels[0].name}):")
    print(f.read())

Nombre de fichiers labels : 28632

Exemple label (plant_a50fd0e4-4f72-4f5b-98d2-7b1ccaacb364___Crnl_L_Mold-6532_180deg_JPG.rf.9e6701329bd0671478a67ba7de6319b1.txt):
0 0.3430078125 0.595703125 0.349140625 0.4248046875


In [8]:
from collections import Counter
from pathlib import Path

labels_path = Path('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/labels')
class_counts = Counter()

for label_file in labels_path.glob('*.txt'):
    with open(label_file, 'r') as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

# Noms des classes
names = [
    'Chlorosis', 'Water Excess', 'Sun Excess', 'Harmful Insects',
    'Water Deficiency', 'Sun Deficiency', 'Powdery Mildew', 'Parasites', 'Abnormal Redness',
    'Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold',
    'Leaf Miner', 'Mosaic Virus', 'Septoria', 'Spider Mites', 'Yellow Leaf Curl',
    'CBB', 'CBSD', 'CGM', 'CMD', 'CASSAVA_HEALTHY'
]

print("📊 Distribution des classes :")
for class_id in sorted(class_counts.keys()):
    name = names[class_id] if class_id < len(names) else f'class_{class_id}'
    print(f"  [{class_id}] {name}: {class_counts[class_id]} annotations")

📊 Distribution des classes :
  [0] Chlorosis: 5067 annotations
  [1] Water Excess: 12073 annotations
  [2] Sun Excess: 15623 annotations
  [3] Harmful Insects: 15342 annotations
  [4] Water Deficiency: 19875 annotations
  [5] Sun Deficiency: 2218 annotations
  [6] Powdery Mildew: 1303 annotations
  [7] Parasites: 2343 annotations
  [8] Abnormal Redness: 750 annotations
  [9] Bacterial Spot: 1917 annotations


In [9]:
import os
import shutil
from pathlib import Path

base = Path('/home/medraedboukari/Desktop/AgriEdge-AI/data')

# Offset pour chaque dataset
offsets = {
    'plant': 0,
    'tomato': 9,
    'cassava': 19
}

for split in ['train', 'valid', 'test']:
    labels_path = base / 'merged' / split / 'labels'
    
    for label_file in labels_path.glob('*.txt'):
        # Identifier le dataset depuis le nom du fichier
        name = label_file.name
        if name.startswith('plant_'):
            offset = offsets['plant']
        elif name.startswith('tomato_'):
            offset = offsets['tomato']
        elif name.startswith('cassava_'):
            offset = offsets['cassava']
        else:
            continue
        
        # Lire et remapper
        with open(label_file, 'r') as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            if line.strip():
                parts = line.strip().split()
                parts[0] = str(int(parts[0]) + offset)
                new_lines.append(' '.join(parts) + '\n')
        
        # Réécrire
        with open(label_file, 'w') as f:
            f.writelines(new_lines)
    
    print(f"✅ {split} remappé")

print("\n✅ Remapping terminé !")

✅ train remappé
✅ valid remappé
✅ test remappé

✅ Remapping terminé !


In [10]:
from collections import Counter
from pathlib import Path

labels_path = Path('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/labels')
class_counts = Counter()

for label_file in labels_path.glob('*.txt'):
    with open(label_file, 'r') as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

names = [
    'Chlorosis', 'Water Excess', 'Sun Excess', 'Harmful Insects',
    'Water Deficiency', 'Sun Deficiency', 'Powdery Mildew', 'Parasites', 'Abnormal Redness',
    'Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold',
    'Leaf Miner', 'Mosaic Virus', 'Septoria', 'Spider Mites', 'Yellow Leaf Curl',
    'CBB', 'CBSD', 'CGM', 'CMD', 'CASSAVA_HEALTHY'
]

print("📊 Distribution des classes après remapping :")
for class_id in sorted(class_counts.keys()):
    name = names[class_id] if class_id < len(names) else f'class_{class_id}'
    print(f"  [{class_id}] {name}: {class_counts[class_id]} annotations")

📊 Distribution des classes après remapping :
  [0] Chlorosis: 813 annotations
  [1] Water Excess: 783 annotations
  [2] Sun Excess: 776 annotations
  [3] Harmful Insects: 1175 annotations
  [4] Water Deficiency: 766 annotations
  [5] Sun Deficiency: 652 annotations
  [6] Powdery Mildew: 868 annotations
  [7] Parasites: 795 annotations
  [8] Abnormal Redness: 642 annotations
  [9] Bacterial Spot: 4254 annotations
  [10] Early Blight: 10647 annotations
  [11] Healthy: 5100 annotations
  [12] Late Blight: 486 annotations
  [13] Leaf Mold: 441 annotations
  [14] Leaf Miner: 1566 annotations
  [15] Mosaic Virus: 435 annotations
  [16] Septoria: 1548 annotations
  [17] Spider Mites: 108 annotations
  [18] Yellow Leaf Curl: 1917 annotations
  [20] CBSD: 643 annotations
  [21] CGM: 9747 annotations
  [22] CMD: 13681 annotations
  [23] CASSAVA_HEALTHY: 18668 annotations


In [11]:
from pathlib import Path

labels_path = Path('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/labels')

# Chercher les fichiers cassava
cassava_files = [f for f in labels_path.glob('cassava_*.txt')]
print(f"Fichiers cassava : {len(cassava_files)}")

# Vérifier les IDs dans quelques fichiers cassava
for f in cassava_files[:5]:
    with open(f, 'r') as file:
        content = file.read().strip()
        if content:
            print(f"{f.name}: {content[:100]}")

Fichiers cassava : 9818
cassava_447053671_jpg.rf.7d243799d8d200e1aa7e6b0486b77cac.txt: 21 1 0.4852841984375 0.9934480625000001 0.40284213125 0.9527140062499999 0.4083010609375 0.926890125
cassava_3161578706_jpg.rf.15968416164206c4a4d0576cf504ef96.txt: 22 0.892578125 0.8268229171874999 0.91015625 0.7747395828125 0.931640625 0.7356770828125 0.935546875
cassava_385391871_jpg.rf.d701811eabcde61a143beed74ad54b47.txt: 22 0.513671875 0.6809895828125 0.51171875 0.64453125 0.52734375 0.61328125 0.544921875 0.53515625 0.
cassava_1359809318_jpg.rf.9a688d7bb9f87b823043900043ce9e25.txt: 23 0.90625 0.6549479171875 0.888671875 0.59765625 0.87109375 0.5638020828125 0.8173828125 0.49739583
cassava_image_271_jpg.rf.108165e69b0f84431dbd26b0f693b41d.txt: 22 0.7795539578125 0.400379375 0.766466934375 0.396700965625 0.779485784375 0.3964737203125 0.789389


In [12]:
from pathlib import Path

def seg_to_bbox(points):
    """Convertit polygone segmentation en bounding box YOLO"""
    xs = points[0::2]
    ys = points[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return x_center, y_center, width, height

labels_path = Path('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/labels')
converted = 0

for label_file in labels_path.glob('cassava_*.txt'):
    with open(label_file, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) > 5:  # segmentation
            class_id = parts[0]
            points = list(map(float, parts[1:]))
            x_c, y_c, w, h = seg_to_bbox(points)
            new_lines.append(f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")
        else:
            new_lines.append(line)
    
    with open(label_file, 'w') as f:
        f.writelines(new_lines)
    converted += 1

print(f"✅ {converted} fichiers cassava convertis seg → bbox")

✅ 9818 fichiers cassava convertis seg → bbox


In [13]:
for split in ['valid', 'test']:
    labels_path = Path(f'/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/{split}/labels')
    converted = 0

    for label_file in labels_path.glob('cassava_*.txt'):
        with open(label_file, 'r') as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 5:
                class_id = parts[0]
                points = list(map(float, parts[1:]))
                x_c, y_c, w, h = seg_to_bbox(points)
                new_lines.append(f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")
            else:
                new_lines.append(line)
        
        with open(label_file, 'w') as f:
            f.writelines(new_lines)
        converted += 1

    print(f"✅ {split}: {converted} fichiers convertis")

✅ valid: 1506 fichiers convertis
✅ test: 1505 fichiers convertis


In [14]:
from collections import Counter
from pathlib import Path

labels_path = Path('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/labels')
class_counts = Counter()

for label_file in labels_path.glob('*.txt'):
    with open(label_file, 'r') as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

names = [
    'Chlorosis', 'Water Excess', 'Sun Excess', 'Harmful Insects',
    'Water Deficiency', 'Sun Deficiency', 'Powdery Mildew', 'Parasites', 'Abnormal Redness',
    'Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold',
    'Leaf Miner', 'Mosaic Virus', 'Septoria', 'Spider Mites', 'Yellow Leaf Curl',
    'CBB', 'CBSD', 'CGM', 'CMD', 'CASSAVA_HEALTHY'
]

print("📊 Distribution finale des classes :")
total = sum(class_counts.values())
for class_id in sorted(class_counts.keys()):
    name = names[class_id] if class_id < len(names) else f'class_{class_id}'
    count = class_counts[class_id]
    pct = count/total*100
    print(f"  [{class_id}] {name}: {count} ({pct:.1f}%)")

print(f"\nTotal annotations : {total}")
print(f"Classes présentes : {len(class_counts)}")

📊 Distribution finale des classes :
  [0] Chlorosis: 813 (1.1%)
  [1] Water Excess: 783 (1.0%)
  [2] Sun Excess: 776 (1.0%)
  [3] Harmful Insects: 1175 (1.5%)
  [4] Water Deficiency: 766 (1.0%)
  [5] Sun Deficiency: 652 (0.9%)
  [6] Powdery Mildew: 868 (1.1%)
  [7] Parasites: 795 (1.0%)
  [8] Abnormal Redness: 642 (0.8%)
  [9] Bacterial Spot: 4254 (5.6%)
  [10] Early Blight: 10647 (13.9%)
  [11] Healthy: 5100 (6.7%)
  [12] Late Blight: 486 (0.6%)
  [13] Leaf Mold: 441 (0.6%)
  [14] Leaf Miner: 1566 (2.0%)
  [15] Mosaic Virus: 435 (0.6%)
  [16] Septoria: 1548 (2.0%)
  [17] Spider Mites: 108 (0.1%)
  [18] Yellow Leaf Curl: 1917 (2.5%)
  [20] CBSD: 643 (0.8%)
  [21] CGM: 9747 (12.7%)
  [22] CMD: 13681 (17.9%)
  [23] CASSAVA_HEALTHY: 18668 (24.4%)

Total annotations : 76511
Classes présentes : 23


In [15]:
yaml_content = """train: /home/medraedboukari/Desktop/AgriEdge-AI/data/merged/train/images
val: /home/medraedboukari/Desktop/AgriEdge-AI/data/merged/valid/images
test: /home/medraedboukari/Desktop/AgriEdge-AI/data/merged/test/images

nc: 23
names: [
  'Chlorosis', 'Water Excess', 'Sun Excess', 'Harmful Insects',
  'Water Deficiency', 'Sun Deficiency', 'Powdery Mildew', 'Parasites', 'Abnormal Redness',
  'Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Leaf Mold',
  'Leaf Miner', 'Mosaic Virus', 'Septoria', 'Spider Mites', 'Yellow Leaf Curl',
  'CBSD', 'CGM', 'CMD', 'CASSAVA_HEALTHY'
]
"""

with open('/home/medraedboukari/Desktop/AgriEdge-AI/data/merged/data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml mis à jour — 23 classes")

✅ data.yaml mis à jour — 23 classes
